# Tier 1 — Grid Search + 5-fold CV (FT-Transformers, SAINT, FT-CUR)

Mesmo protocolo da versão LSSVM, **só os modelos PyTorch** que rodam mais rápido em GPU:
- 4 baselines: FT-Softmax, FT-TopK, FT-Entmax, FT-Sparsemax
- SAINT (inter-instâncias denso)
- FT-CUR (inter-instâncias comprimido via Nyström)

**Grade reduzida** (decidido com orientador): só `num_blocks/n_layers` e `num_heads/n_heads` (+ `m_ratio` no FT-CUR). Demais hiperparâmetros fixados nos defaults do Gorishniy 2021 (`lr=1e-3, batch_size=256, dropout=0.1, max_epochs=15, patience=3`).

**SAINT + FT-CUR**: `early_stop_metric='val_loss'` (correção da H2 ablation, $p<10^{-5}$).

**Antes de rodar:** `Runtime → Change runtime type → T4 GPU`. Com Colab Pro tente L4 ou A100 se aparecer.

**Resume robusto:** restaura do Drive, sync background a cada 5 min, save final ao terminar.

**Notebook complementar:** `tier1_gridcv_lssvm_colab.ipynb` (LSSVMs no CPU).

In [ ]:
# ── Célula 1: Verifica runtime GPU ──────────────────────────────────────────
import torch
print(f'CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Compute: {torch.cuda.get_device_capability(0)}')
    print(f'Memória: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('GPU não detectada — vá em Runtime > Change runtime type > T4 GPU')

In [ ]:
# ── Célula 2: Clonar do GitHub ──────────────────────────────────────────────
import os
PROJECT_DIR = '/content/sparse-lssvm-transformers-study'
GIT_URL = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'

if os.path.exists(PROJECT_DIR):
    !cd {PROJECT_DIR} && git pull --rebase
else:
    !git clone {GIT_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
!git log --oneline -3
print(f'\nDiretório atual: {os.getcwd()}')

In [ ]:
# ── Célula 3: Dependências ──────────────────────────────────────────────────
!pip install -q numpy scipy scikit-learn pandas xlrd pyarrow entmax einops
# torch já vem instalado no Colab GPU runtime

import numpy, scipy, sklearn, torch, entmax
print(f'numpy {numpy.__version__} | scipy {scipy.__version__} | '
      f'sklearn {sklearn.__version__} | torch {torch.__version__} | '
      f'entmax {entmax.__version__}')

In [ ]:
# ── Célula 4: Baixar datasets Tier 1 ────────────────────────────────────────
!python scripts/download_data.py --tier 1
!ls -lh data/raw/ | head -15

In [ ]:
# ── Célula 5: Montar Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/dissertacao_tier1_gridcv'
import os
os.makedirs(DRIVE_PATH, exist_ok=True)
print(f'Drive em: {DRIVE_PATH}')
!ls -lh '{DRIVE_PATH}' 2>/dev/null

In [ ]:
# ── Célula 6: Restaurar progresso (resume do Drive) ─────────────────────────
import shutil
from pathlib import Path

OUTPUT_FNAME = 'tier1_gridcv_transformers.json'
drive_results = Path(DRIVE_PATH)
local_results = Path('results')
local_results.mkdir(exist_ok=True)

src = drive_results / OUTPUT_FNAME
dst = local_results / OUTPUT_FNAME
if src.exists():
    shutil.copy(src, dst)
    import json
    n = len(json.load(open(dst)))
    print(f'✓ Restaurado: {OUTPUT_FNAME} ({n} entries)')
else:
    print(f'• Começando do zero: {OUTPUT_FNAME}')

In [ ]:
# ── Célula 7: Sync para Drive em background (a cada 5 min) ─────────────────
%%writefile /content/sync_to_drive.sh
#!/bin/bash
while true; do
    sleep 300
    cp -u /content/sparse-lssvm-transformers-study/results/tier1_gridcv_transformers.json \
          "$1/tier1_gridcv_transformers.json" 2>/dev/null
done

In [ ]:
import subprocess
sync_proc = subprocess.Popen(['bash', '/content/sync_to_drive.sh', DRIVE_PATH])
print(f'Sync rodando em background (PID {sync_proc.pid}) — salva a cada 5 min')

In [ ]:
# ── Célula 8: Inspecionar a grade ───────────────────────────────────────────
from src.tuning.grids import GRIDS, grid_size

TRANSFORMER_MODELS = [
    'FTTransformer_softmax', 'FTTransformer_topk',
    'FTTransformer_entmax', 'FTTransformer_sparsemax',
    'SAINTColnorm',
    'FTTransformerCURColnorm',
]

print(f'{"Modelo":<28}{"Pontos":>8}{"Fits/seed/dataset":>20}')
print('-' * 60)
total_fits = 0
for m in TRANSFORMER_MODELS:
    g = grid_size(m)
    print(f'{m:<28}{g:>8}{g*5:>20}')
    total_fits += g * 5 * 9 * 30
print(f'\nTotal: ~{total_fits:,} fits (6 modelos × 9 datasets × 30 seeds)')
print(f'Estimativa T4: ~{total_fits * 3.6 / 3600:.0f} h')

In [ ]:
# ── Célula 9: Rodar tudo ────────────────────────────────────────────────────
# Script escreve em results/ (local). Sync da Célula 7 copia pro Drive a cada 5 min.
# Se a sessão cair, reabra e re-execute desde a Célula 1 — restaura tudo do Drive.

models_str = ' '.join(TRANSFORMER_MODELS)
datasets_str = 'BCW PID HAB VCP GCR AUS AI4I TWS TWM TWC'

!python -u scripts/run_tier1_gridcv.py \
    --models {models_str} \
    --datasets {datasets_str} \
    --output results/tier1_gridcv_transformers.json \
    --log-level INFO 2>&1 | tee /tmp/tier1_transformers_run.log

In [ ]:
# ── Célula 10: Save final no Drive ──────────────────────────────────────────
import shutil, signal
from pathlib import Path

try:
    sync_proc.send_signal(signal.SIGTERM)
except Exception:
    pass

drive_dest = Path(DRIVE_PATH)
src = Path('results') / OUTPUT_FNAME
dst = drive_dest / OUTPUT_FNAME
if src.exists():
    shutil.copy(src, dst)
    print(f'✓ Salvo: {dst.name} ({src.stat().st_size / 1024:.1f} KB)')

print(f'\nConteúdo do Drive:')
!ls -lh '{DRIVE_PATH}'

In [ ]:
# ── Célula 11: Resumo dos resultados ────────────────────────────────────────
import json
from collections import defaultdict
import statistics as st

records = json.load(open('results/' + OUTPUT_FNAME))
print(f'Total records: {len(records)}\n')

agg = defaultdict(lambda: defaultdict(list))
for r in records:
    if r.get('status') != 'ok':
        continue
    agg[r['variant']][r['dataset']].append(r['test_f1_macro'])

datasets = 'BCW PID HAB VCP GCR AUS AI4I TWS TWM TWC'.split()
header = ['Modelo'] + datasets + ['Média']
print(f'{header[0]:<28}' + ''.join(f'{h:>7}' for h in header[1:]))
print('-' * (28 + 7 * len(header[1:])))
for variant in TRANSFORMER_MODELS:
    line = [variant]
    means = []
    for d in datasets:
        vals = agg[variant][d]
        if vals:
            m = st.mean(vals)
            line.append(f'{m:>7.3f}')
            means.append(m)
        else:
            line.append(f'{"-":>7}')
    avg = st.mean(means) if means else 0
    line.append(f'{avg:>7.3f}')
    print(f'{line[0]:<28}' + ''.join(line[1:]))